# Avazu Click Prediction : Data Prep

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
# samples made in 01_eda , every 40th row of each file
train = pd.read_parquet('../../data/ctr/train_sample.parquet')
valid = pd.read_parquet('../../data/ctr/valid_sample.parquet')
test = pd.read_parquet('../../data/ctr/test_sample.parquet')
len(train) , len(valid) , len(test)

(808580, 101073, 101073)

## 3. Machine Learning
### Split : the given train / valid / test files are used as they are.
### Train is for fitting , valid for choosing settings (threshold , C) , test is used once at the end.

### Feature engineering : hour (YYMMDDHH) becomes hour of day and weekday.
### Feature selection , dropping. Cost = how much validation log loss rises without the column (table in 05_sample_size_and_columns) :
### id : a row key.
### device_id , device_ip : hundreds of thousands of values , they identify users , not a kind of impression.
### C1 : it is close to device_type (V = 0.82 in 01_eda). Saves 5 columns , costs 0.00001.
### C14 : it fixes C15 , C16 , C17 and C21 (V = 1 in 01_eda). It has more detail than C17 , but it takes 421 columns , costs 0.00094 to leave out , and the model doesn't fit in the GPU with it.
### C18 , C19 , C21 : C17 fixes or almost fixes them (V = 0.99 , 0.87 , 1). Saves 108 columns , costs 0.00013 , and makes room for 1 in 40.
### app_domain : every app_id has exactly one app_domain. Saves 24 columns , costs 0.00006.

In [3]:
drop = ['id' , 'hour' , 'device_id' , 'device_ip' , 'C14' , 'C1' , 'app_domain' , 'C18' , 'C19' , 'C21']

def features(d):
    d = d.copy()
    d['hour_of_day'] = d['hour'].str[6:]
    d['weekday'] = pd.to_datetime(d['hour'] , format='%y%m%d%H').dt.day_name()
    return d.drop(columns=drop)

train , valid , test = features(train) , features(valid) , features(test)
cat_cols = [c for c in train.columns if c != 'click']
cat_cols

['banner_pos',
 'site_id',
 'site_domain',
 'site_category',
 'app_id',
 'app_category',
 'device_model',
 'device_type',
 'device_conn_type',
 'C15',
 'C16',
 'C17',
 'C20',
 'hour_of_day',
 'weekday']

### Rare values : site_id , app_id and device_model have thousands of values , most seen a handful of times.
### Values in fewer than 0.025% of train rows become 'other'. They are picked from train only , so valid and test don't leak in.
### Why 0.025% : at 0.1% , 30% of device_model rows land in other , at 0.025% only 13%. Log loss falls by 0.0043 at 1 in 32 (05_sample_size_and_columns).
### Sample and cut are limited together by the GPU , and 1 in 40 at 0.025% is what fits.

In [4]:
keep = {}
for col in cat_cols:
    share = train[col].value_counts(normalize=True)
    keep[col] = share[share >= 0.00025].index

for d in [train , valid , test]:
    for col in cat_cols:
        d[col] = d[col].where(d[col].isin(keep[col]) , 'other')

pd.DataFrame({'values left' : train[cat_cols].nunique() , 'share of rows in other' : (train[cat_cols] == 'other').mean().round(3)})

,values left,share of rows in other
banner_pos,5,0.000
site_id,183,0.058
site_domain,147,0.047
site_category,14,0.000
app_id,127,0.045
app_category,14,0.001
device_model,488,0.131
device_type,4,0.000
device_conn_type,4,0.000
C15,5,0.000


### Encoding : one hot. The categories come from train , so all three splits get the same columns.
### drop_first=True removes one column per feature since it can be worked out from the others.

In [5]:
for col in cat_cols:
    levels = sorted(train[col].unique())
    for d in [train , valid , test]:
        d[col] = pd.Categorical(d[col] , categories=levels)

def encode(d):
    X = pd.get_dummies(d[cat_cols] , drop_first=True , dtype='float32')
    return X , d['click']

X_train , y_train = encode(train)
X_val , y_val = encode(valid)
X_test , y_test = encode(test)
X_train.shape , X_val.shape , X_test.shape

((808580, 1356), (101073, 1356), (101073, 1356))

### Scaling : standardising every column with train values only.
### A rare value's column is almost always 0 , so without this gradient descent barely moves its weight.

In [6]:
scaler = StandardScaler().fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train) , columns=X_train.columns , dtype='float32')
X_val = pd.DataFrame(scaler.transform(X_val) , columns=X_train.columns , dtype='float32')
X_test = pd.DataFrame(scaler.transform(X_test) , columns=X_train.columns , dtype='float32')
X_train.iloc[: , :5].describe().round(2)

,banner_pos_1,banner_pos_2,banner_pos_7,banner_pos_other,site_id_05c65e53
count,808580.00,808580.00,808580.00,808580.00,808580.00
mean,-0.00,-0.00,-0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00
min,-0.62,-0.02,-0.03,-0.02,-0.02
25%,-0.62,-0.02,-0.03,-0.02,-0.02
50%,-0.62,-0.02,-0.03,-0.02,-0.02
75%,1.61,-0.02,-0.03,-0.02,-0.02
max,1.61,53.63,30.90,50.90,57.56


In [7]:
X_train.to_parquet('../../data/ctr/X_train.parquet')
X_val.to_parquet('../../data/ctr/X_val.parquet')
X_test.to_parquet('../../data/ctr/X_test.parquet')
y_train.to_frame().to_parquet('../../data/ctr/y_train.parquet')
y_val.to_frame().to_parquet('../../data/ctr/y_val.parquet')
y_test.to_frame().to_parquet('../../data/ctr/y_test.parquet')